# TRACE Pipeline

**TRACE** (Temporal Recovery Analysis via Community Engagement) predicts NBA player recovery outcomes from Achilles tendon injuries using social-media sentiment signals collected over the year following each injury.

The pipeline runs five sequential stages:

| Stage | Script | Reads | Writes |
|-------|--------|-------|--------|
| 1 | `merge_datasets.py` | `llm_classifications_full.csv`, `sentiment_results.csv` | `trace_filtered_dataset.csv` |
| 2 | `compute_outcomes.py` | `player_stats_raw.csv` | `player_outcomes.csv` |
| 3 | `build_episodes.py` | `trace_filtered_dataset.csv`, `player_outcomes.csv` | `injury_episodes.csv` |
| 4 | `build_sequences.py` | `injury_episodes.csv`, `player_outcomes.csv` | `temporal_sequences.csv` |
| 5 | `train_classifier.py` | `temporal_sequences.csv` | `loo_predictions.csv`, `trace_classifier.pkl` |

Run all cells top-to-bottom to reproduce results from scratch.

In [1]:
import subprocess
import sys
import os
from pathlib import Path
import pandas as pd

# Resolve project root relative to this notebook's location
PROJECT_ROOT = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
# Fallback: set explicitly if needed
# PROJECT_ROOT = Path("/path/to/TRACE")

os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

PYTHON = sys.executable
print(f"Python interpreter: {PYTHON}")


def run_script(script_path: str) -> None:
    """Run a Python script and stream its stdout/stderr to the cell output."""
    result = subprocess.run(
        [PYTHON, script_path],
        capture_output=True,
        text=True,
        cwd=PROJECT_ROOT,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("--- STDERR ---", flush=True)
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{script_path} exited with code {result.returncode}")
    print(f"\n✓ {script_path} completed successfully.")

Working directory: /Users/sathwiktoduru/Library/CloudStorage/GoogleDrive-vedanth.sathwik@gmail.com/My Drive/GT/Masters/TRACE
Python interpreter: /Users/sathwiktoduru/Library/CloudStorage/GoogleDrive-vedanth.sathwik@gmail.com/My Drive/GT/Masters/TRACE/TRACE/trace/bin/python


### Stage 1: Merge datasets

`merge_datasets.py` joins two upstream CSVs on their shared positional row index:

- **`llm_classifications_full.csv`** — Gemini LLM labels (`SUITABLE` / `UNSUITABLE`) for every scraped post.
- **`sentiment_results.csv`** — FinBERT sentiment scores and scraper metadata for the same posts.

A record is kept if it is **`SUITABLE`** (LLM) **or** `is_achilles_related` (scraper flag), then duplicates are dropped.

**Output:** `runners/data/trace_filtered_dataset.csv`

In [3]:
run_script("TRACE/runners/merge_datasets.py")

=== llm_classifications_full.csv ===
Shape: (24467, 11)
Columns: ['row_index', 'source_platform', 'is_achilles_related', 'text_preview', 'classification', 'confidence', 'reasoning', 'recovery_phase', 'key_entities', 'error', 'processed_at']

   row_index source_platform  is_achilles_related                                                                      text_preview classification  confidence                                                                                                                                                                                                                                                                                                                                                           reasoning recovery_phase                                                                                                                               key_entities                                                                                         e

### Stage 2: Compute player outcome labels

`compute_outcomes.py` reads `player_stats_raw.csv` (pre- and post-injury PER, days missed, games played in return season) and applies a three-part rule to assign a binary recovery label:

**Outcome = 1** only when **all** of the following hold:
- `days_out ≤ 540` — returned within ~18 months
- `post_injury_per ≥ 0.80 × pre_injury_per` — retained ≥ 80 % of prior form (PER)
- `games_return_season ≥ 25` — played a meaningful number of games

Otherwise **outcome = 0**.

**Output:** `runners/data/player_outcomes.csv`

In [6]:
run_script("TRACE/runners/compute_outcomes.py")

🔍 Loaded player_stats_raw: (27, 4)

Player : kevin_durant
  Pre-injury seasons : 2017-2018 (PER 26.0), 2018-2019 (PER 24.2)
  Pre-injury avg PER : 25.10
  Return season      : 2020-2021  →  PER 26.4, 32 G
  Injury date        : 2019-06-10
  Days out           : 479  ✅ (<= 730)
  PER retention      : 26.4 / 17.57  ✅ (>= 80% of 25.10)
  Games played       : 32  ✅ (>= 30)
  OUTCOME            : 1 (success)
------------------------------------------------------------------------
Player : brandon_jennings
  Pre-injury seasons : 2012-2013 (PER 16.1), 2013-2014 (PER 15.6)
  Pre-injury avg PER : 15.85
  Return season      : 2015-2016  →  PER 13.7, 48 G
  Injury date        : 2015-01-25
  Days out           : 249  ✅ (<= 730)
  PER retention      : 13.7 / 11.10  ✅ (>= 80% of 15.85)
  Games played       : 48  ✅ (>= 30)
  OUTCOME            : 1 (success)
------------------------------------------------------------------------
Player : wesley_matthews
  Pre-injury seasons : 2012-2013 (PER 14.1), 20

### Stage 3: Build injury episodes

`build_episodes.py` constructs a per-player episode dataset by:

1. Loading `trace_filtered_dataset.csv` and `player_outcomes.csv`.
2. For each player, finding all records whose `created_date` falls within **±365 days** of their known injury date **and** whose text explicitly mentions the player by name (regex match).
3. Computing `days_from_injury` (signed integer) for each matched record.
4. Dropping players with fewer than 10 matched records (insufficient signal).

**Output:** `runners/data/injury_episodes.csv`

In [7]:
run_script("TRACE/runners/build_episodes.py")

🔍 Loaded trace_filtered_dataset: (4520, 45)
🔍 Loaded player_outcomes:        (9, 7)

=== Date range diagnostics ===
  filtered created_date : 2012-02-07 08:00:00+00:00  →  2026-03-04 01:48:50+00:00
  outcomes injury_date  : 2015-01-25 00:00:00+00:00  →  2020-11-18 00:00:00+00:00

=== kevin_durant deep-dive ===
  injury_date : 2019-06-10 00:00:00+00:00
  window      : 2018-06-10 00:00:00+00:00  →  2020-06-09 00:00:00+00:00
  records in date window (no name filter) : 854
  records mentioning Durant/Kevin (no date filter) : 1,037
  records passing BOTH filters : 369

Player                     Records  Status
-------------------------------------------------------
kevin_durant                   369  ✅ included
brandon_jennings                85  ✅ included
wesley_matthews                 78  ✅ included
rudy_gay                       106  ✅ included
demarcus_cousins               222  ✅ included
jj_barea                        69  ✅ included
john_wall                      180  ✅ included
r

### Stage 4: Build temporal sequences

`build_sequences.py` converts the flat episode records into a structured time series:

1. Keeps only the **post-injury window** (`days_from_injury` in [0, 365]).
2. Assigns each record to one of **52 weekly bins** (`bin_index = days_from_injury // 7`).
3. Aggregates within each bin:
   - `avg_sentiment_positive`, `avg_sentiment_neutral`, `avg_sentiment_negative`
   - `total_engagement` (sum of engagement scores)
   - `record_count`
4. Fills all 52 bins for every player — bins with no data are zero-padded.
5. Joins the binary outcome label from `player_outcomes.csv`.

**Output:** `runners/data/temporal_sequences.csv` — 52 rows per player

In [8]:
run_script("TRACE/runners/build_sequences.py")


  injury_episodes.csv  (injury_episodes.csv)
Columns (47):
  row_index, source_platform_llm, is_achilles_related_llm, text_preview, classification, confidence, reasoning, recovery_phase_llm, key_entities, error, processed_at, source_platform_sent, source_detail, author, url, text_content, created_date, engagement_score, engagement_secondary, engagement_tier, relevance_score, recovery_phase_sent, mentioned_players, is_achilles_related_sent, is_quality_content, text_length, year, month, year_month, num_comments_extracted, avg_comment_score, total_comment_words, num_replies_extracted, avg_reply_likes, total_reply_words, body_word_count, fetch_success, uploaded_at, sentiment_label, sentiment_score, sentiment_positive, sentiment_negative, sentiment_neutral, finbert_model_version, analyzed_at, player_name, days_from_injury

First 2 rows:
 row_index source_platform_llm  is_achilles_related_llm                                                                     text_preview classification  co

### Stage 5: Train classifier

`train_classifier.py` builds and evaluates a logistic regression classifier:

**Feature engineering (212 features per player):**
- Flatten 4 sentiment/engagement signals × 52 bins = **208 time-series features**
- 4 summary statistics over non-zero bins: `mean_positive`, `mean_negative`, `total_records`, `non_zero_bins`

**Evaluation — Leave-One-Out Cross-Validation (LOO-CV):**  
With only 9 players, standard train/test splits waste too much data. LOO-CV trains on 8 players and predicts the 9th, repeating for every player. This gives the most honest estimate of generalisation with a small cohort.

**Outputs:**
- `runners/data/loo_predictions.csv` — per-player predictions
- `runners/data/trace_classifier.pkl` — final model trained on all players

In [9]:
run_script("TRACE/runners/train_classifier.py")


  temporal_sequences.csv
Shape: (468, 8)

Outcome distribution:
  outcome=0: 3 player(s)
  outcome=1: 6 player(s)

  Feature matrix
X shape: (9, 212)  (players × features)
y shape: (9,)

Players in dataset:
  [0] brandon_jennings                outcome=1
  [1] demarcus_cousins                outcome=0
  [2] jj_barea                        outcome=0
  [3] john_wall                       outcome=1
  [4] kevin_durant                    outcome=1
  [5] klay_thompson                   outcome=1
  [6] rodney_hood                     outcome=0
  [7] rudy_gay                        outcome=1
  [8] wesley_matthews                 outcome=1

  LOO-CV Results
Accuracy: 0.667 (6/9 correct)

Player                           True   Pred  Correct
-------------------------------------------------------
brandon_jennings                    1      1  ✓
demarcus_cousins                    0      1  ✗
jj_barea                            0      1  ✗
john_wall                           1      1  ✓
kevin_dur

### Results summary

The cells below load the saved predictions and player outcomes to produce a consolidated result table, overall accuracy, and a text confusion matrix.

In [ ]:
DATA_DIR = PROJECT_ROOT / "TRACE" / "runners" / "data"

preds = pd.read_csv(DATA_DIR / "loo_predictions.csv")
outcomes = pd.read_csv(DATA_DIR / "player_outcomes.csv")

# ── Merged result table ────────────────────────────────────────────────────
summary = preds[["player_name", "true_outcome", "predicted_outcome", "correct"]].copy()
summary["correct"] = summary["correct"].map({True: "✓", False: "✗"})
summary.columns = ["Player", "True outcome", "Predicted", "Correct"]

print("=" * 60)
print("  Per-player LOO-CV predictions")
print("=" * 60)
print(summary.to_string(index=False))

# ── Overall accuracy ───────────────────────────────────────────────────────
n_correct = (preds["correct"] == True).sum()
n_total = len(preds)
accuracy = n_correct / n_total
print(f"\nOverall LOO-CV Accuracy: {n_correct}/{n_total}  ({accuracy:.1%})")

# ── Confusion matrix ───────────────────────────────────────────────────────
labels = sorted(preds["true_outcome"].unique())
print(f"\nConfusion matrix  (rows = true, cols = predicted)")
print(f"Labels: {labels}")
header = "         " + "".join(f"  pred={l}" for l in labels)
print(header)
for true_l in labels:
    row_str = f"true={true_l}  "
    for pred_l in labels:
        count = ((preds["true_outcome"] == true_l) & (preds["predicted_outcome"] == pred_l)).sum()
        row_str += f"  {count:>6}"
    print(row_str)

# ── Highlight wrong predictions ────────────────────────────────────────────
wrong = preds[preds["correct"] == False]
if wrong.empty:
    print("\nAll predictions correct.")
else:
    print(f"\nMis-classified ({len(wrong)}):")
    for _, r in wrong.iterrows():
        print(f"  {r['player_name']}: true={r['true_outcome']}, predicted={r['predicted_outcome']}")

SyntaxError: invalid syntax (122279400.py, line 1)